In [263]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import string
import os
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import shutil
import gc

In [264]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [265]:
# Hyperparameters

EMBEDDING_DIMENSION = 32
LR = 0.001
EPOCHS = 10
BATCH_SIZE = 32

## 3 Sequence-to-Sequence Modeling

### Data Preprocessing

In [266]:
data = pd.read_csv('topical_chat_pairs.csv', sep='\t')
data.head()

,id,conversation_id,message,answer
0,0,1,Are you a fan of Google or Microsoft?,Both are excellent technology they are helpfu...
1,1,1,Both are excellent technology they are helpfu...,"I'm not a huge fan of Google, but I use it a..."
2,2,1,"I'm not a huge fan of Google, but I use it a...",Google provides online related services and p...
3,3,1,Google provides online related services and p...,"Yeah, their services are good. I'm just not a..."
4,4,1,"Yeah, their services are good. I'm just not a...",Google is leading the alphabet subsidiary and...


In [267]:
word2index = {}
index2word = {}

word2index['<SOS>'] = 0
word2index['<EOS>'] = 1
word2index['<PAD>'] = 2
index2word[0] = '<SOS>'
index2word[1] = '<EOS>'
index2word[2] = '<PAD>'

In [268]:
lenghts = []
unique_words = set()
for text in data['message']:
    tokens = word_tokenize(text.strip().lower())
    lenghts.append(len(tokens))
    for word in tokens:
        unique_words.add(word)
for text in data['answer']:
    tokens = word_tokenize(text.strip().lower())
    lenghts.append(len(tokens))
    for word in tokens:
        unique_words.add(word)
print(f'Number of unique words: {len(unique_words)}')
print(f'{sorted(list(unique_words))[:5]}')
AVERAGE_LENGTH = round(np.mean(lenghts))
print(f'Average length of sentences: {AVERAGE_LENGTH}')

Number of unique words: 41112
['!', '#', '$', '%', '&']
Average length of sentences: 23


In [269]:
for word in unique_words:
    index = len(word2index)
    word2index[word] = index
    index2word[index] = word
print(f'Number of words in vocabulary: {len(word2index)}')

Number of words in vocabulary: 41115


In [270]:
def collate_fn(batch):
    messages, answers = zip(*batch)
    preprocessed_messages = []
    for message in messages:
        message = word_tokenize(message.strip().lower())
        preprocessed_message = []
        preprocessed_message.append(word2index['<SOS>'])
        for word in message[:AVERAGE_LENGTH]:
            preprocessed_message.append(word2index[word])
        preprocessed_message.append(word2index['<EOS>'])
        if len(preprocessed_message) < AVERAGE_LENGTH + 2:  # +2 for <SOS> and <EOS>
            preprocessed_message += [word2index['<PAD>']] * (AVERAGE_LENGTH + 2 - len(preprocessed_message))
        preprocessed_messages.append(preprocessed_message)

    max_length = max(len(text) for text in answers)
    preprocessed_answers = []
    for answer in answers:
        answer = word_tokenize(answer.strip().lower())
        preprocessed_answer = []
        preprocessed_answer.append(word2index['<SOS>'])
        for word in answer[:AVERAGE_LENGTH]:
            preprocessed_answer.append(word2index[word])
        preprocessed_answer.append(word2index['<EOS>'])
        if len(preprocessed_answer) < AVERAGE_LENGTH + 2:  # +2 for <SOS> and <EOS>
            preprocessed_answer += [word2index['<PAD>']] * (AVERAGE_LENGTH + 2 - len(preprocessed_answer))
        preprocessed_answers.append(preprocessed_answer)

    preprocessed_messages = torch.tensor(preprocessed_messages, dtype=torch.long)
    preprocessed_answers = torch.tensor(preprocessed_answers, dtype=torch.long)
    return preprocessed_messages, preprocessed_answers

In [271]:
class SequenceDataset(Dataset):
    def __init__(self, messages, answers):
        if len(messages) != len(answers):
            raise ValueError("Messages and answers must have the same length.")
        if not isinstance(messages, list) or not isinstance(answers, list):
            try:
                messages = messages.tolist()
                answers = answers.tolist()
            except AttributeError:
                raise TypeError("Messages and answers must be convertible to lists.")
        self.messages = messages
        self.answers = answers

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        return self.messages[idx], self.answers[idx]

In [272]:
dataset = SequenceDataset(data['message'], data['answer'])

In [273]:
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=True)

In [274]:
for messages, answers in train_loader:
    print(f'Messages batch shape: {messages.shape}')
    print(f'Answers batch shape: {answers.shape}')
    for i in range(4):
        print(f'Message {i}: {messages[i]}')
        print(f'Answer {i}: {answers[i]}')
    break

Messages batch shape: torch.Size([32, 25])
Answers batch shape: torch.Size([32, 25])
Message 0: tensor([    0, 14488, 15777, 24554, 15321, 35942, 10373, 10859,  8452, 17389,
        10915, 28212, 32005,  8452,     1,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2])
Answer 0: tensor([    0, 22330,  8208, 21031,  3732,  9175,  1370, 10843,  2239, 31961,
        38852, 22330,  1082, 24877, 35288, 29754, 18639,  2239, 19922, 37933,
        39810, 14195, 17157,     1,     2])
Message 1: tensor([    0, 14488,  8067, 35765,  8452,  8067,  1754, 41005, 15597, 21535,
        17398, 39900,     1,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2])
Answer 1: tensor([    0, 14488, 41005, 15597,  9827, 29589, 32612, 32194, 10843, 21161,
        31503,  8452, 10859, 22273,  6377, 38160,     1,     2,     2,     2,
            2,     2,     2,     2,     2])
Message 2: tensor([    0, 14488, 21816, 21835,  1047, 41005, 36556,  7019

### Model

In [275]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input):
        embedded = self.embedding(input)
        # h0 = torch.zeros(1, BATCH_SIZE, self.hidden_size, device=embedded.device)
        h0 = torch.zeros(1, input.shape[0], self.hidden_size, device=embedded.device)
        _, output = self.gru(embedded, h0)
        return output

In [276]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.linear = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=2)

    def forward(self, input, hidden):
        embedded = self.embedding(input)
        embedded = self.relu(embedded)
        embedded = embedded.unsqueeze(0)
        output, last_hidden = self.gru(embedded, hidden)
        output = self.linear(output)
        output = self.softmax(output)
        output = output.squeeze(0)
        return output, last_hidden

In [277]:
encoder = EncoderRNN(len(word2index), EMBEDDING_DIMENSION).to(DEVICE)
decoder = DecoderRNN(EMBEDDING_DIMENSION, len(word2index)).to(DEVICE)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr=LR)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=LR)
criterion = nn.NLLLoss(ignore_index=word2index['<PAD>'])

if not os.path.exists('models'):
    os.makedirs('models')
encoder_path = 'models/encoder.pth'
decoder_path = 'models/decoder.pth'

In [278]:
def validate_model(encoder, decoder, val_loader, criterion, device=DEVICE):
    encoder.eval()
    decoder.eval()
    total_loss = 0.0

    with torch.no_grad():
        val_loader = tqdm(val_loader, desc="Validating")
        for i, (messages, answers) in enumerate(val_loader):
            messages = messages.to(device)
            answers = answers.to(device)

            encoder_output = encoder(messages)
            last_hidden = encoder_output
            decoder_input = answers[:, 0]  # Start with the first token of the answer
            batch_loss = 0.0
            for j in range(1, answers.size(1)):
                output, last_hidden = decoder(decoder_input, last_hidden)
                loss = criterion(output, answers[:, j])
                batch_loss += loss.item()
                decoder_input = torch.argmax(output, dim=1)  # Use the predicted token as the next input
            total_loss += batch_loss / answers.size(1)
    avg_loss = total_loss / len(val_loader)
    return avg_loss

In [279]:
def train(encoder, decoder, train_loader, val_loader, criterion, encoder_optimizer, decoder_optimizer, epochs, device):
    writer = SummaryWriter()
    average_training_loss_per_epoch = []
    validation_loss_per_epoch = []
    min_validation_loss = float('inf')
    for epoch in range(epochs):
        encoder.train()
        decoder.train()
        total_loss = 0.0
        train_loader = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}')
        for i, (messages, answers) in enumerate(train_loader):
            messages, answers = messages.to(device), answers.to(device)
            encoder_optimizer.zero_grad()
            decoder_optimizer.zero_grad()

            encoder_output = encoder(messages)
            last_hidden = encoder_output
            batch_loss = 0
            for j in range(1, answers.size(1)):
                output, last_hidden = decoder(answers[:, j-1], last_hidden)
                loss = criterion(output, answers[:, j])
                batch_loss += loss
                # decoder_input = torch.argmax(output, dim=1) # uncomment to not use teacher forcing

            batch_loss = batch_loss / answers.size(1)
            batch_loss.backward()
            encoder_optimizer.step()
            decoder_optimizer.step()

            total_loss += batch_loss.item()
            train_loader.set_postfix({"loss": total_loss / (i + 1)})

        validation_loss = validate_model(encoder, decoder, val_loader, criterion, device)

        average_training_loss_per_epoch.append(total_loss / len(train_loader))
        validation_loss_per_epoch.append(validation_loss)

        if validation_loss < min_validation_loss:
            min_validation_loss = validation_loss
            torch.save(encoder.state_dict(), encoder_path)
            torch.save(decoder.state_dict(), decoder_path)
            print(f"Validation loss improved to {min_validation_loss:.4f}, models saved.")

        writer.add_scalar('Training Loss', total_loss / len(train_loader), epoch)
        writer.add_scalar('Validation Loss', validation_loss, epoch)
        writer.flush()
    writer.close()
        

In [280]:
train(
    encoder=encoder,
    decoder=decoder,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    encoder_optimizer=encoder_optimizer,
    decoder_optimizer=decoder_optimizer,
    epochs=EPOCHS,
    device=DEVICE
)

Validating: 100%|██████████| 561/561 [00:06<00:00, 85.68it/s]


Validation loss improved to 6.7725, models saved.


Validating: 100%|██████████| 561/561 [00:06<00:00, 83.44it/s]


In [281]:
encoder.load_state_dict(torch.load(encoder_path))
decoder.load_state_dict(torch.load(decoder_path))

<All keys matched successfully>

In [282]:
def print_test_samples(encoder, decoder, test_loader, n=5, t=AVERAGE_LENGTH, device=DEVICE):
    encoder.eval()
    decoder.eval()
    total_loss = 0.0

    random_indices = np.random.choice(np.arange(len(test_loader)), size=n, replace=False)
    print(f'Randomly selected indices for test samples: {sorted(random_indices)}')

    with torch.no_grad():
        for i, (messages, answers) in enumerate(test_loader):
            if i not in random_indices:
                continue
            print(f'\nSample {i}:')
            random_message_index = np.random.choice(np.arange(len(messages)), size=1, replace=False).item()
            random_message = messages[random_message_index].to(device)
            print(f"Message: {' '.join([index2word[index.to('cpu').item()] for index in random_message if index not in [0, 1, 2]])}")

            messages = messages.to(device)
            answer = []

            encoder_output = encoder(random_message.unsqueeze(0))  # Add batch dimension
            last_hidden = encoder_output
            decoder_input = torch.tensor([word2index['<SOS>']]).to(device)  # Start with the first token of the answer
            for j in range(1, t):
                output, last_hidden = decoder(decoder_input, last_hidden)
                decoder_input = torch.argmax(output, dim=1)  # Use the predicted token as the next input
                word = index2word[decoder_input.item()]
                if word == '<EOS>':
                    break
                answer.append(word)
            print(f'Predicted answer: {" ".join(answer)}')

In [283]:
print_test_samples(encoder, decoder, test_loader, n=5, t=AVERAGE_LENGTH, device=DEVICE)

Randomly selected indices for test samples: [175, 230, 380, 446, 517]

Sample 175:
Message: surprise dancers seem to abound ! i love to watch capable people do that !
Predicted answer: i do n't know that . i wonder if they are a lot of the first time .

Sample 230:
Message: got ta gain it back ! i 'm a poor skater too , maybe field hockey would be okay to play .
Predicted answer: i do n't know that . i wonder if they are a lot of the first time .

Sample 380:
Message: have you ever seen the version of soccer that has 3 teams ?
Predicted answer: i do n't know that . i wonder if they are a lot of the first time .

Sample 446:
Message: honestly , i do n't really follow the ceremonies like the golden globe , but hearing it now ya its a shame
Predicted answer: i do n't know that . i wonder if they are a lot of the first time .

Sample 517:
Message: i bet costs are reduced with volume , do you like to play video games ?
Predicted answer: i do n't know that . i wonder if they are a lot of the